#House Price Prediction Model

##Importing Data

In [1]:
import pandas as pd
import pyodbc

db_file_path = r'C:\Projekt\HousePriceML\data\HousePriceML.accdb'
conn = pyodbc.connect(r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=' + db_file_path)

cursor = conn.cursor()

cursor.execute('SELECT * FROM Housing')
columns = [column[0] for column in cursor.description]
data = cursor.fetchall()
rows = cursor.fetchall()

for row in rows:
    print(row)

df = pd.DataFrame.from_records(data, columns=columns)
cursor.close()
conn.close()


In [2]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,"""7229300521""","""20141013T000000""",231300.0,2,1.00,1180,5650,"""1""",0,0,...,7,1180,0,1955,0,"""98178""",47.5112,-122.257,1340,5650
1,"""6414100192""","""20141209T000000""",538000.0,3,2.25,2570,7242,"""2""",0,0,...,7,2170,400,1951,1991,"""98125""",47.7210,-122.319,1690,7639
2,"""5631500400""","""20150225T000000""",180000.0,2,1.00,770,10000,"""1""",0,0,...,6,770,0,1933,0,"""98028""",47.7379,-122.233,2720,8062
3,"""2487200875""","""20141209T000000""",604000.0,4,3.00,1960,5000,"""1""",0,0,...,7,1050,910,1965,0,"""98136""",47.5208,-122.393,1360,5000
4,"""1954400510""","""20150218T000000""",510000.0,3,2.00,1680,8080,"""1""",0,0,...,8,1680,0,1987,0,"""98074""",47.6168,-122.045,1800,7503


##Data Clean-up

In [3]:
df = df.drop('id', axis=1)

In [4]:
df['date'] = df['date'].str.strip('"')
df['floors'] = df['floors'].str.strip('"')
df['zipcode'] = df['zipcode'].str.strip('"')

In [5]:
df['floors'] = df['floors'].astype(float)
df['zipcode'] = df['zipcode'].astype(int)

In [6]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

In [7]:
df['date'] = df['date'].dt.year

In [8]:
df[['bedrooms', 'bathrooms']] = round(df[['bedrooms', 'bathrooms']], 0)

##Feature Engineering

In [9]:
df['age'] = df['date'] - df['yr_built']
df['bed_per_bath'] = df['bedrooms'] / (df['bathrooms'] + 1)
df['time_since_renovated'] = df['date'] - df['yr_renovated']
df['rooms'] = df['bathrooms'] + df['bedrooms']
df['sqft_per_room'] = df['sqft_living'] / (df['rooms'] + 1)

In [10]:
df = df.iloc[:, [0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,1]]

In [11]:
df.head()

,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,lat,long,sqft_living15,sqft_lot15,age,bed_per_bath,time_since_renovated,rooms,sqft_per_room,price
0,2014,2,1.0,1180,5650,1.0,0,0,3,7,...,47.5112,-122.257,1340,5650,59,1.0,2014,3.0,295.000000,231300.0
1,2014,3,2.0,2570,7242,2.0,0,0,3,7,...,47.7210,-122.319,1690,7639,63,1.0,23,5.0,428.333333,538000.0
2,2015,2,1.0,770,10000,1.0,0,0,3,6,...,47.7379,-122.233,2720,8062,82,1.0,2015,3.0,192.500000,180000.0
3,2014,4,3.0,1960,5000,1.0,0,0,5,7,...,47.5208,-122.393,1360,5000,49,1.0,2014,7.0,245.000000,604000.0
4,2015,3,2.0,1680,8080,1.0,0,0,3,8,...,47.6168,-122.045,1800,7503,28,1.0,2015,5.0,280.000000,510000.0


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  21613 non-null  int32  
 1   bedrooms              21613 non-null  int64  
 2   bathrooms             21613 non-null  float64
 3   sqft_living           21613 non-null  int64  
 4   sqft_lot              21613 non-null  int64  
 5   floors                21613 non-null  float64
 6   waterfront            21613 non-null  int64  
 7   view                  21613 non-null  int64  
 8   condition             21613 non-null  int64  
 9   grade                 21613 non-null  int64  
 10  sqft_above            21613 non-null  int64  
 11  sqft_basement         21613 non-null  int64  
 12  yr_built              21613 non-null  int64  
 13  yr_renovated          21613 non-null  int64  
 14  zipcode               21613 non-null  int64  
 15  lat                   21613 no

##Dataset Split

In [13]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
Y = df['price']

In [14]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [15]:
x_train.info()

<class 'pandas.DataFrame'>
Index: 17290 entries, 5596 to 21095
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  17290 non-null  int32  
 1   bedrooms              17290 non-null  int64  
 2   bathrooms             17290 non-null  float64
 3   sqft_living           17290 non-null  int64  
 4   sqft_lot              17290 non-null  int64  
 5   floors                17290 non-null  float64
 6   waterfront            17290 non-null  int64  
 7   view                  17290 non-null  int64  
 8   condition             17290 non-null  int64  
 9   grade                 17290 non-null  int64  
 10  sqft_above            17290 non-null  int64  
 11  sqft_basement         17290 non-null  int64  
 12  yr_built              17290 non-null  int64  
 13  yr_renovated          17290 non-null  int64  
 14  zipcode               17290 non-null  int64  
 15  lat                   17290 non-

##XGBoost

In [16]:
import xgboost as xgb

model = xgb.XGBRegressor(n_estimators = 1500, learning_rate = 0.06, max_depth = 5)

#Training
model.fit (x_train, y_train, eval_set=[(x_test, y_test)], verbose=100)

#Prediction
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {mse}")

[0]	validation_0-rmse:364934.85662
[100]	validation_0-rmse:138076.42118
[200]	validation_0-rmse:132538.79813
[300]	validation_0-rmse:131264.05057
[400]	validation_0-rmse:129776.67068
[500]	validation_0-rmse:129491.62671
[600]	validation_0-rmse:129151.43420
[700]	validation_0-rmse:128709.17528
[800]	validation_0-rmse:128319.11107
[900]	validation_0-rmse:128194.97475
[1000]	validation_0-rmse:128010.05571
[1100]	validation_0-rmse:128035.58001
[1200]	validation_0-rmse:127918.59137
[1300]	validation_0-rmse:127896.96190
[1400]	validation_0-rmse:127950.57747
[1499]	validation_0-rmse:127906.09473
Mean Squared Error: 16359969100.96203


##Results


In [17]:
import numpy as np

results = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_pred,
    'Absolute Error': np.abs(y_test - y_pred),
    'Percentage Error': np.abs((y_test - y_pred) / y_test) * 100
})

formatted_results = results.copy()

formatted_results['Actual Price'] = formatted_results['Actual Price'].map('${:,.2f}'.format)
formatted_results['Predicted Price'] = formatted_results['Predicted Price'].map('${:,.2f}'.format)
formatted_results['Absolute Error'] = formatted_results['Absolute Error'].map('${:,.2f}'.format)
formatted_results['Percentage Error'] = formatted_results['Percentage Error'].map('{:,.2f}%'.format)

In [18]:
formatted_results.head(20)

,Actual Price,Predicted Price,Absolute Error,Percentage Error
4700,"$670,000.00","$644,766.31","$25,233.69",3.77%
9648,"$359,950.00","$355,606.31","$4,343.69",1.21%
6613,"$545,000.00","$500,295.81","$44,704.19",8.20%
3324,"$555,000.00","$603,403.06","$48,403.06",8.72%
11268,"$1,400,000.00","$1,206,828.38","$193,171.62",13.80%
3718,"$1,240,000.00","$1,129,469.12","$110,530.88",8.91%
17003,"$500,000.00","$460,774.38","$39,225.62",7.85%
5073,"$859,000.00","$791,079.00","$67,921.00",7.91%
443,"$350,000.00","$360,323.62","$10,323.62",2.95%
18368,"$500,000.00","$499,632.22",$367.78,0.07%


In [19]:
total_percentage_error = np.mean(results['Percentage Error'])
median_ape = np.median(results['Percentage Error'])

print(f"Median Percentage Error: {median_ape:.2f}%")
print(f'Mean Percentage Error: {total_percentage_error:.2f}%')

Median Percentage Error: 8.06%
Mean Percentage Error: 11.94%


In [20]:
formatted_results['Actual Price'].str.strip("$")
formatted_results['Predicted Price'].str.strip("$")

4700       644,766.31
9648       355,606.31
6613       500,295.81
3324       603,403.06
11268    1,206,828.38
             ...     
16659      517,412.28
5225       295,075.41
8063       268,615.22
16615      658,197.94
4833       556,388.81
Name: Predicted Price, Length: 4323, dtype: str

In [21]:
formatted_results

,Actual Price,Predicted Price,Absolute Error,Percentage Error
4700,"$670,000.00","$644,766.31","$25,233.69",3.77%
9648,"$359,950.00","$355,606.31","$4,343.69",1.21%
6613,"$545,000.00","$500,295.81","$44,704.19",8.20%
3324,"$555,000.00","$603,403.06","$48,403.06",8.72%
11268,"$1,400,000.00","$1,206,828.38","$193,171.62",13.80%
...,...,...,...,...
16659,"$552,321.00","$517,412.28","$34,908.72",6.32%
5225,"$299,000.00","$295,075.41","$3,924.59",1.31%
8063,"$260,000.00","$268,615.22","$8,615.22",3.31%
16615,"$665,000.00","$658,197.94","$6,802.06",1.02%


In [22]:
formatted_results.to_csv("formatted_results.csv")